In [78]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("bro it is working")


from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

bro it is working


Root chain

In [79]:
template_capital = ChatPromptTemplate.from_messages([
    ("system", "you are a hrlpful assistant"),
    ("human","what is the capital of : {input}")
])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_capital  = template_capital | llm_model | str_parser 

In [80]:
def chain_function(final_answer:str):
    return {"text" : final_answer}

chain_function_runnable = RunnableLambda(chain_function)

Parallel chain 1

In [81]:
template_LinkedInpost = ChatPromptTemplate.from_messages([
    ("system","you are a social media post generator."),
    ("human", "create a small post for the following text for LinkedIn: {text}")
])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_linkedIn = template_LinkedInpost | llm_model | str_parser

Parallel chain 2

In [82]:
template_Instapost = ChatPromptTemplate.from_messages([
("system","you are a social media post generator."),
("human", "create a small post for the following text for Instagram: {text}")])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_Instagram = template_Instapost | llm_model | str_parser

Final Orchestration of Root chian, Parallel Chain 1 and 2

In [83]:
final_chain = (
    chain_capital |
    chain_function_runnable |
    RunnableParallel(branches = {"linkedin" : chain_linkedIn, "instagram" : chain_Instagram })
)

In [ ]:
#final_chain.invoke("Karnataka")

{'branches': {'linkedin': 'Did you know? Bengaluru — formerly Bangalore — is the capital of Karnataka.\n\nOnce known as the “Garden City,” Bengaluru has transformed into India’s leading tech and innovation hub, home to startups, multinational R&D centers, and a deep talent pool. Whether you’re exploring expansion, hiring, or collaboration, Bengaluru remains a strategic city to watch.\n\nWhat’s your experience working in or with Bengaluru? Share your thoughts below.\n\n#Bengaluru #Karnataka #TechEcosystem #Startups #Innovation',
  'instagram': 'Here are three ready-to-use Instagram caption options (pick one) plus hashtags and an alt-text suggestion.\n\nOption 1 — Simple & factual\nBengaluru (formerly Bangalore) — the capital of Karnataka. 🇮🇳🏙️\n\nOption 2 — Engaging / conversational\nDid you know? Bengaluru (formerly Bangalore) is the capital of Karnataka! Have you been? ✨🌆 Drop your favorite Bengaluru memory below. ⤵️\n\nOption 3 — Travel-style\nBengaluru — Karnataka’s capital, formerl

Chain as a Runnable

In [84]:
def beautify(final_answer:dict):
    linkedIn_repsonse = final_answer['branches']['linkedin']
    instagram_reposnse = final_answer['branches']['instagram']

    return {"linkedin" : linkedIn_repsonse, "instagram" : instagram_reposnse }

beautify_runnable = RunnableLambda(beautify)

Final chain

In [85]:
template_StrongPost = ChatPromptTemplate.from_messages([
        ("system", "you are a social media content reviewer"),
        ("human", "Review these two posts and give one best post from these two post. LinkedIn  post: {linkedin} Instagram post : {instagram}")
])

llm_model = init_chat_model(model="gpt-5-mini")

str_parser = StrOutputParser()

chain_bestPost = template_StrongPost | llm_model | str_parser

In [86]:
Result_chain = final_chain | beautify_runnable | chain_bestPost

Result_chain.invoke("Karnataka")

'Best choice: the LinkedIn post.\n\nWhy:\n- Tone and content match the LinkedIn audience — professional, focused on technology and entrepreneurship.\n- It provides personal connection (“Proud to be connected…”) which builds credibility and engagement on a professional network.\n- It uses relevant hashtags without relying on emojis (more appropriate for LinkedIn).\n\nSuggested small polish for clarity and engagement:\n"Bengaluru (formerly Bangalore) is the capital of Karnataka — a vibrant blend of culture and innovation. Proud to be part of this thriving hub of technology and entrepreneurship. #Bengaluru #Karnataka #India #Tech #Startups"\n\nIf you want the same message for Instagram instead, use the shorter, emoji-friendly version you already have.'